# %DATE% %DESCRIPTION%

# Set up (Collapse and run)

## Imports

#### Built-ins

In [ ]:
import os
import sys
from pathlib import Path
import glob
import shutil
import re
from typing import List, Union, Optional

#### Common

In [ ]:
#Plotting Libraries

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.subplots as sp

In [ ]:
import ipynbname

### Test Functions

In [ ]:
import pandas as pd
import numpy as np

def generate_dfs():
    """
    Returns a list of functions. Each function, when called, returns a 
    pandas DataFrame containing synthetic jet engine performance data.
    All DataFrames conform to an identical column schema using ALL CAPS short notation.
    """

    def get_sls_part_power():
        """Generates a Sea Level Static (SLS) part-power line (Idle to MTO)."""
        TRA = np.linspace(0, 100, 25)
        
        N1 = 22 + 78 * (TRA / 100)**0.8
        FNT = 30000 * (N1 / 100)**3.5
        EGT = 350 + 600 * (TRA / 100)**1.5
        SFC = 0.55 - 0.15 * (TRA / 100) + 0.1 * (TRA / 100)**2 
        WFT = FNT * SFC
        
        return pd.DataFrame({
            'TITLE': 'SLS Part-Power Line',
            'RATING': 'SLS_PART_POWER',
            'ALT': 0.0,
            'XM': 0.0,
            'TRA': TRA,
            'N1': N1,
            'EGT': EGT,
            'FNT': FNT,
            'WFT': WFT,
            'SFC': SFC
        })

    def get_cruise_part_power():
        """Generates a Cruise part-power line (35,000 ft, 0.8 Mach)."""
        TRA = np.linspace(0, 100, 25)
        ALT = 35000.0
        XM = 0.8
        
        alt_lapse = np.exp(-ALT / 25000)
        mach_effect = (1 - 0.3 * XM + 0.5 * XM**2)
        
        N1 = 25 + 75 * (TRA / 100)**0.85
        FNT = 30000 * (N1 / 100)**3.5 * alt_lapse * mach_effect
        EGT = 300 + 650 * (TRA / 100)**1.5 
        SFC = 0.50 - 0.10 * (TRA / 100) + 0.08 * (TRA / 100)**2 
        WFT = FNT * SFC
        
        return pd.DataFrame({
            'TITLE': 'Cruise Part-Power Line',
            'RATING': 'CRUISE_PART_POWER',
            'ALT': ALT,
            'XM': XM,
            'TRA': TRA,
            'N1': N1,
            'EGT': EGT,
            'FNT': FNT,
            'WFT': WFT,
            'SFC': SFC
        })

    def get_mto_envelope_sweep():
        """Generates an Altitude-Mach sweep at Max Takeoff (MTO) power."""
        altitudes = np.linspace(0, 40000, 20)
        machs = np.linspace(0.0, 0.9, 10)
        
        ALT, XM = np.meshgrid(altitudes, machs)
        ALT = ALT.flatten()
        XM = XM.flatten()
        
        FNT = 30000 * np.exp(-ALT / 25000) * (1 - 0.35 * XM + 0.6 * XM**2)
        SFC = 0.55 + 0.000002 * ALT + 0.05 * XM
        WFT = FNT * SFC
        
        # MTO sweep assumes 100% Throttle, 100% N1, and a fixed peak EGT for array broadcast
        TRA = np.full_like(ALT, 100.0)
        N1 = np.full_like(ALT, 100.0)
        EGT = np.full_like(ALT, 950.0)
        
        return pd.DataFrame({
            'TITLE': 'MTO Envelope Sweep',
            'RATING': 'MTO_SWEEP',
            'ALT': ALT,
            'XM': XM,
            'TRA': TRA,
            'N1': N1,
            'EGT': EGT,
            'FNT': FNT,
            'WFT': WFT,
            'SFC': SFC
        })

    def get_mcl_envelope_sweep():
        """Generates an Altitude-Mach sweep at Max Climb (MCL) power."""
        altitudes = np.linspace(0, 40000, 20)
        machs = np.linspace(0.0, 0.9, 10)
        
        ALT, XM = np.meshgrid(altitudes, machs)
        ALT = ALT.flatten()
        XM = XM.flatten()
        
        FNT = 26000 * np.exp(-ALT / 25000) * (1 - 0.35 * XM + 0.6 * XM**2)
        SFC = 0.52 + 0.000002 * ALT + 0.04 * XM
        WFT = FNT * SFC
        
        # MCL sweep assumes ~85% Throttle, ~90% N1, and cooler EGTs
        TRA = np.full_like(ALT, 85.0)
        N1 = np.full_like(ALT, 90.4) 
        EGT = np.full_like(ALT, 820.0)
        
        return pd.DataFrame({
            'TITLE': 'MCL Envelope Sweep',
            'RATING': 'MCL_SWEEP',
            'ALT': ALT,
            'XM': XM,
            'TRA': TRA,
            'N1': N1,
            'EGT': EGT,
            'FNT': FNT,
            'WFT': WFT,
            'SFC': SFC
        })

    return [
        get_sls_part_power,
        get_cruise_part_power,
        get_mto_envelope_sweep,
        get_mcl_envelope_sweep
    ]

## Paths

In [ ]:
this_filepath = ipynbname.path()
github_path = Path(r"%GITHUB_DIR%")

if os.path.exists(github_path):
    sys.path.append(str(github_path))
else:
    raise FileNotFoundError("The specified GitHub path does not exist. Please check the path and try again.")

## Local Imports

File Creation and Manipulation

In [ ]:
from toms_utils.file_manipulation import open_directory, tree, ls, mkdir, cd, pwd, find, rm, mv, cp, grep, sed, du, archive, extract, cat, head, tail
from Auto_PPTX_Album.autoppt import create_ppt
from toms_utils.unichart import UnichartNotebook as UC

In [ ]:
import os
from pathlib import Path

def get_notebook_location():
    try:
        from IPython import get_ipython
        ipython = get_ipython()
        
        if ipython is not None and hasattr(ipython, 'kernel'):
            # Get the current working directory (where notebook executes)
            return Path(os.getcwd())
        else:
            return Path(os.getcwd())
            
    except Exception as e:
        print(f"Could not determine notebook location: {e}")
        return Path(os.getcwd())

# Usage
notebook_location = get_notebook_location()
print(f"Notebook location: {notebook_location}")

### Finance

In [ ]:
# from fredapi import Fred
# from linux_profile.secrets import fred_api_key as FRED_API_KEY
# fred = Fred(api_key=FRED_API_KEY)
# data = fred.get_series('SP500')

In [ ]:
def scatter_subplots(df_list, x, y_list, subplot_titles=None):
    fig = sp.make_subplots(rows=1, cols=len(y_list), subplot_titles=subplot_titles or y_list)
    for i, y in enumerate(y_list):
        for df in df_list:
            scatter = px.scatter(df, x=x, y=y)
            for trace in scatter.data:
                fig.add_trace(trace, row=1, col=i+1)
    fig.update_layout(height=400, width=300*len(y_list), showlegend=False)
    fig.show()

# Main Body

In [ ]:
cd(r"%WORKING_DIRECTORY%")
print(os.getcwd())
print(this_filepath)